<a href="https://colab.research.google.com/github/tiendungnguyenw0907-boop/DocumentIntelligence/blob/main/_notebooks/2020-06-05-PDFs%20In%20Python%20using%20PyMuPDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Layer 1. Physical Understanding**

## 1. Ingestion

> *Tiếp nhận tài liệu từ các nguồn khác nhau và đánh giá chất lượng đầu vào nhằm bảo đảm tài liệu đủ điều kiện để đưa vào quy trình xử lý tự động. Đây là bước kiểm soát chất lượng đầu vào của toàn bộ hệ thống*



### 1.1 Tiếp nhận tài liệu

### 1.2 Kiểm tra chất lượng

# 2. Physical Analysis

> *Phân tích các đặc tính vật lý và kỹ thuật của tài liệu nhằm hiểu tài liệu được tạo ra như thế nào, gồm những thành phần gì và cần áp dụng chiến lược xử lý nào. Kết quả của bước này giúp hệ thống xác định phương pháp OCR, nhận dạng bố cục và trích xuất thông tin phù hợp*



## 2.1. PDF Parsing & Format Analysis

### 2.1.1. Phân tích cấu trúc PDF

#### 2.1.1.1 Installation

In [3]:
# !pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 67.9 MB/s eta 0:00:00


#### 2.1.1.2. UPLOAD PDF




In [1]:
from google.colab import files

uploaded = files.upload()

pdf_file = list(uploaded.keys())[0]

print("PDF:", pdf_file)

Saving SachBCTHKQKT2021.pdf to SachBCTHKQKT2021.pdf
PDF: SachBCTHKQKT2021.pdf


#### 2.1.1.3. PDF Structure Analyzer

In [4]:
import fitz
import json
import os

from tqdm import tqdm
from collections import Counter


class PDFStructureAnalyzer:

    def __init__(self, pdf_path):

        self.pdf_path = pdf_path

        self.doc = fitz.open(pdf_path)

    def analyze(self):

        file_size_mb = round(
            os.path.getsize(self.pdf_path) / 1024 / 1024,
            2
        )

        result = {

            "document_id": "DOC001",

            "file_info": {},

            "document_structure": {},

            "object_statistics": {},

            "font_statistics": [],

            "page_structure": [],

            "page_summary": {},

            "technical_flags": {}

        }

        result["file_info"] = {

            "file_name": os.path.basename(self.pdf_path),

            "file_size_mb": file_size_mb,

            "pdf_version": "Unknown",

            "encrypted": self.doc.needs_pass,

            "page_count": len(self.doc)

        }

        total_text_objects = 0

        total_image_objects = 0

        total_annotation_objects = 0

        total_vector_objects = 0

        font_counter = Counter()

        pages_with_text = 0

        pages_without_text = 0

        pages_with_images = 0

        pages_with_annotations = 0

        rotated_pages = 0

        has_bookmark = False

        has_hyperlink = False

        has_text_layer = False

        has_image_layer = False

        page_structure = []

        print("Analyzing pages...")

        for page_num in tqdm(range(len(self.doc))):

            page = self.doc[page_num]

            try:

                text = page.get_text()

            except:

                text = ""

            try:

                text_dict = page.get_text("dict")

                blocks = text_dict.get("blocks", [])

                text_object_count = len(blocks)

            except:

                text_object_count = 0

            try:

                image_count = len(page.get_images(full=True))

            except:

                image_count = 0

            try:

                fonts = page.get_fonts()

            except:

                fonts = []

            font_names = []

            for f in fonts:

                if len(f) > 3:

                    font_name = str(f[3])

                    font_names.append(font_name)

                    font_counter[font_name] += 1

            annotation_count = 0

            try:

                annots = page.annots()

                if annots:

                    annotation_count = len(list(annots))

            except:

                pass

            try:

                links = page.get_links()

                if len(links) > 0:

                    has_hyperlink = True

            except:

                pass

            page_info = {

                "page_number": page_num + 1,

                "has_text_layer": len(text.strip()) > 0,

                "text_object_count": text_object_count,

                "image_object_count": image_count,

                "font_count": len(set(font_names)),

                "annotation_count": annotation_count,

                "page_rotation": page.rotation

            }

            page_structure.append(page_info)

            total_text_objects += text_object_count

            total_image_objects += image_count

            total_annotation_objects += annotation_count

            if len(text.strip()) > 0:

                pages_with_text += 1

                has_text_layer = True

            else:

                pages_without_text += 1

            if image_count > 0:

                pages_with_images += 1

                has_image_layer = True

            if annotation_count > 0:

                pages_with_annotations += 1

            if page.rotation != 0:

                rotated_pages += 1

        try:

            toc = self.doc.get_toc()

            has_bookmark = len(toc) > 0

        except:

            pass

        result["document_structure"] = {

            "has_text_layer": has_text_layer,

            "has_image_layer": has_image_layer,

            "has_annotation": pages_with_annotations > 0,

            "has_form_field": False,

            "has_embedded_font": len(font_counter) > 0,

            "has_vector_graphic": False,

            "has_bookmark": has_bookmark,

            "has_hyperlink": has_hyperlink,

            "has_watermark": False

        }

        result["object_statistics"] = {

            "total_text_objects": total_text_objects,

            "total_image_objects": total_image_objects,

            "total_font_objects": len(font_counter),

            "total_annotation_objects": total_annotation_objects,

            "total_vector_objects": total_vector_objects

        }

        font_statistics = []

        for font_name, count in font_counter.most_common():

            font_statistics.append({

                "font_name": font_name,

                "usage_count": count

            })

        result["font_statistics"] = font_statistics

        result["page_structure"] = page_structure

        result["page_summary"] = {

            "pages_with_text_layer": pages_with_text,

            "pages_without_text_layer": pages_without_text,

            "pages_with_images": pages_with_images,

            "pages_with_annotations": pages_with_annotations,

            "rotated_pages": rotated_pages

        }

        result["technical_flags"] = {

            "contains_scanned_pages":
                pages_without_text > 0,

            "contains_native_pages":
                pages_with_text > 0,

            "contains_mixed_pages":
                pages_with_text > 0 and pages_without_text > 0,

            "contains_large_images":
                total_image_objects > 100,

            "contains_rotated_pages":
                rotated_pages > 0

        }

        return result

#### 2.1.1.4. Chạy phân tích

In [5]:
analyzer = PDFStructureAnalyzer(pdf_file)

result = analyzer.analyze()

print("Done")

Analyzing pages...


100%|██████████| 284/284 [00:03<00:00, 91.01it/s]

Done


#### 2.1.1.5. Xuất JSON

In [6]:
output_file = "pdf_structure_profile.json"

with open(
    output_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        result,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", output_file)

Saved: pdf_structure_profile.json


#### 2.1.1.6. Download JSON

In [7]:
from google.colab import files

files.download("pdf_structure_profile.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### 2.1.2. Xác định loại PDF

#### 2.1.2.1 Load PDF Structure Profile

In [ ]:
import json

with open("/content/pdf_structure_profile.json", "r", encoding="utf-8") as f:
    structure_profile = json.load(f)

print("Loaded")

Loaded


#### 2.1.2.2 PDF Type Analyzer

In [10]:
class PDFTypeAnalyzer:

    def __init__(self, structure_profile):

        self.structure_profile = structure_profile

    def analyze(self):

        page_structure = self.structure_profile["page_structure"]

        total_pages = len(page_structure)

        native_pages = []
        scan_pages = []
        mixed_pages = []

        for page in page_structure:

            has_text = page["has_text_layer"]

            image_count = page["image_object_count"]

            if has_text and image_count == 0:

                native_pages.append(
                    page["page_number"]
                )

            elif (not has_text) and image_count > 0:

                scan_pages.append(
                    page["page_number"]
                )

            else:

                mixed_pages.append(
                    page["page_number"]
                )

        native_count = len(native_pages)

        scan_count = len(scan_pages)

        mixed_count = len(mixed_pages)

        native_ratio = round(
            native_count / total_pages * 100,
            2
        )

        scan_ratio = round(
            scan_count / total_pages * 100,
            2
        )

        mixed_ratio = round(
            mixed_count / total_pages * 100,
            2
        )

        # ==========================
        # Determine PDF Type
        # ==========================

        if scan_count == 0 and mixed_count == 0:

            pdf_type = "NATIVE"

        elif native_count == 0 and mixed_count == 0:

            pdf_type = "SCAN"

        else:

            pdf_type = "HYBRID"

        # ==========================
        # OCR Strategy
        # ==========================

        if pdf_type == "NATIVE":

            ocr_required = False

            ocr_mode = "NONE"

        elif pdf_type == "SCAN":

            ocr_required = True

            ocr_mode = "FULL_DOCUMENT"

        else:

            ocr_required = True

            ocr_mode = "PAGE_BASED"

        # ==========================
        # Layout Strategy
        # ==========================

        if pdf_type == "NATIVE":

            layout_strategy = "TEXT_FIRST"

        elif pdf_type == "SCAN":

            layout_strategy = "OCR_FIRST"

        else:

            layout_strategy = "HYBRID"

        # ==========================
        # Processing Strategy
        # ==========================

        if pdf_type == "NATIVE":

            processing_strategy = "DIRECT_LAYOUT"

        elif pdf_type == "SCAN":

            processing_strategy = "OCR_LAYOUT"

        else:

            processing_strategy = "MIXED"

        result = {

            "pdf_type": pdf_type,

            "page_count": total_pages,

            "native_pages": native_count,

            "scan_pages": scan_count,

            "mixed_pages": mixed_count,

            "native_ratio": native_ratio,

            "scan_ratio": scan_ratio,

            "mixed_ratio": mixed_ratio,

            "ocr_required": ocr_required,

            "ocr_mode": ocr_mode,

            "layout_strategy": layout_strategy,

            "processing_strategy": processing_strategy,

            "page_groups": {

                "native_pages": native_pages,

                "scan_pages": scan_pages,

                "mixed_pages": mixed_pages

            }

        }

        return result

#### 2.1.2.3 Run

In [11]:
analyzer = PDFTypeAnalyzer(
    structure_profile
)

pdf_type_profile = analyzer.analyze()

print(
    json.dumps(
        pdf_type_profile,
        ensure_ascii=False,
        indent=2
    )
)

{
  "pdf_type": "NATIVE",
  "page_count": 284,
  "native_pages": 284,
  "scan_pages": 0,
  "mixed_pages": 0,
  "native_ratio": 100.0,
  "scan_ratio": 0.0,
  "mixed_ratio": 0.0,
  "ocr_required": false,
  "ocr_mode": "NONE",
  "layout_strategy": "TEXT_FIRST",
  "processing_strategy": "DIRECT_LAYOUT",
  "page_groups": {
    "native_pages": [
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9,
      10,
      11,
      12,
      13,
      14,
      15,
      16,
      17,
      18,
      19,
      20,
      21,
      22,
      23,
      24,
      25,
      26,
      27,
      28,
      29,
      30,
      31,
      32,
      33,
      34,
      35,
      36,
      37,
      38,
      39,
      40,
      41,
      42,
      43,
      44,
      45,
      46,
      47,
      48,
      49,
      50,
      51,
      52,
      53,
      54,
      55,
      56,
      57,
      58,
      59,
      60,
      61,
      62,
      63,
      64,
      65,
      66,
      6

#### 2.1.2.4 Save JSON

In [12]:
with open(
    "/content/pdf_type_profile.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        pdf_type_profile,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved")

Saved


### 2.1.3. Phân tích nội dung tài liệu

#### 2.1.3.1. Code

In [8]:
import fitz
import json


class DocumentContentAnalyzer:

    def __init__(
        self,
        pdf_file,
        structure_profile,
        pdf_type_profile
    ):

        self.pdf_file = pdf_file

        self.structure_profile = structure_profile

        self.pdf_type_profile = pdf_type_profile

        self.doc = fitz.open(pdf_file)

    def analyze(self):

        contains_text = False
        contains_image = False
        contains_table = False
        contains_form = False
        contains_hyperlink = False
        contains_bookmark = False
        contains_signature = False
        contains_stamp = False

        pages_with_text = 0
        pages_with_images = 0
        estimated_table_pages = 0
        pages_with_links = 0
        pages_with_forms = 0

        # =====================
        # Bookmark
        # =====================

        try:

            toc = self.doc.get_toc()

            if len(toc) > 0:

                contains_bookmark = True

        except Exception:

            pass

        # =====================
        # Page Analysis
        # =====================

        for page in self.doc:

            # -------------------
            # Text
            # -------------------

            text = page.get_text()

            if text.strip():

                contains_text = True

                pages_with_text += 1

            # -------------------
            # Images
            # -------------------

            images = page.get_images(
                full=True
            )

            if len(images) > 0:

                contains_image = True

                pages_with_images += 1

            # -------------------
            # Hyperlinks
            # -------------------

            links = page.get_links()

            if len(links) > 0:

                contains_hyperlink = True

                pages_with_links += 1

            # -------------------
            # Forms
            # -------------------

            try:

                widgets = page.widgets()

                if widgets:

                    contains_form = True

                    pages_with_forms += 1

            except Exception:

                pass

            # -------------------
            # Table Estimation
            #
            # Rule-based
            # -------------------

            text_dict = page.get_text(
                "dict"
            )

            block_count = len(
                text_dict.get(
                    "blocks",
                    []
                )
            )

            if block_count >= 20:

                estimated_table_pages += 1

                contains_table = True

        result = {

            "document_content_profile": {

                "contains_text":
                    contains_text,

                "contains_image":
                    contains_image,

                "contains_table":
                    contains_table,

                "contains_form":
                    contains_form,

                "contains_hyperlink":
                    contains_hyperlink,

                "contains_bookmark":
                    contains_bookmark,

                "contains_signature":
                    contains_signature,

                "contains_stamp":
                    contains_stamp
            },

            "content_statistics": {

                "pages_with_text":
                    pages_with_text,

                "pages_with_images":
                    pages_with_images,

                "estimated_table_pages":
                    estimated_table_pages,

                "pages_with_links":
                    pages_with_links,

                "pages_with_forms":
                    pages_with_forms
            }
        }

        return result

#### 2.1.3.2. Run

In [13]:
with open(
    "pdf_structure_profile.json",
    "r",
    encoding="utf-8"
) as f:

    structure_profile = json.load(f)

with open(
    "pdf_type_profile.json",
    "r",
    encoding="utf-8"
) as f:

    pdf_type_profile = json.load(f)

analyzer = DocumentContentAnalyzer(
    pdf_file,
    structure_profile,
    pdf_type_profile
)

content_profile = analyzer.analyze()

print(
    json.dumps(
        content_profile,
        ensure_ascii=False,
        indent=2
    )
)

{
  "document_content_profile": {
    "contains_text": true,
    "contains_image": false,
    "contains_table": true,
    "contains_form": true,
    "contains_hyperlink": false,
    "contains_bookmark": false,
    "contains_signature": false,
    "contains_stamp": false
  },
  "content_statistics": {
    "pages_with_text": 284,
    "pages_with_images": 0,
    "estimated_table_pages": 184,
    "pages_with_links": 0,
    "pages_with_forms": 284
  }
}


#### 2.1.3.3. Lưu file

In [14]:
with open(
    "document_content_profile.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        content_profile,
        f,
        ensure_ascii=False,
        indent=2
    )

## 2.2. Layout Pre-Detection